# Relevant Python 3.11 Changes — Advanced Problems with Solutions

This notebook is a practical, advanced-level tour of the Python 3.11 changes that are most useful in production code.

It emphasizes:

- executable examples rather than feature lists,
- failure modes and debugging techniques,
- best-practice patterns,
- advanced exercises with complete solutions,
- tests and assertions that make the solutions self-checking.

> **Target runtime:** Python 3.11 or newer. The notebook intentionally avoids third-party dependencies.

## Learning objectives

By the end of this notebook, you should be able to:

1. use fine-grained traceback positions to diagnose failures faster;
2. report multiple independent failures with `ExceptionGroup` and handle them with `except*`;
3. enrich errors using `BaseException.add_note()`;
4. apply structured concurrency with `asyncio.TaskGroup` and `asyncio.timeout()`;
5. parse and validate TOML configuration using `tomllib`;
6. use Python 3.11 typing tools such as `Self`, `Required`, `NotRequired`, `LiteralString`, `Never`, and variadic generics;
7. model string constants with `StrEnum`;
8. control regex backtracking with atomic groups and possessive quantifiers;
9. use signed-zero formatting and the integer-string conversion safety limit;
10. inspect CPython 3.11's specializing adaptive interpreter;
11. combine several features in a realistic capstone.

Official references:

- [What’s New in Python 3.11](https://docs.python.org/3.11/whatsnew/3.11.html)
- [PEP 657 — Fine-grained error locations](https://peps.python.org/pep-0657/)
- [PEP 654 — Exception Groups and `except*`](https://peps.python.org/pep-0654/)
- [PEP 678 — Exception notes](https://peps.python.org/pep-0678/)
- [Python 3.11 typing documentation](https://docs.python.org/3.11/library/typing.html)

## Notebook conventions and best practices

- Examples use deterministic data and bounded waits.
- Errors are caught only when the example is specifically demonstrating them.
- Exercises state observable requirements instead of prescribing one implementation.
- Solutions include assertions or tests.
- `except*` handlers are used only for grouped failures; ordinary `except` remains the default for ordinary exceptions.
- Type hints are treated as static-analysis metadata unless explicitly inspected at runtime.

In [1]:
import asyncio
import dis
import json
import math
import re
import sys
import timeit
import tomllib
import traceback
from dataclasses import dataclass, fields
from decimal import Decimal
from enum import StrEnum, auto
from io import BytesIO
from typing import (
    Generic,
    LiteralString,
    Never,
    NotRequired,
    Required,
    Self,
    TypeVarTuple,
    TypedDict,
    Unpack,
    assert_never,
    get_type_hints,
)

print(sys.version)
if sys.version_info < (3, 11):
    raise RuntimeError('This notebook requires Python 3.11 or newer.')

3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


# 1. PEP 657 — Fine-grained error locations in tracebacks

Python 3.11 code objects contain more precise source-position information. Tracebacks can underline the exact expression that failed rather than only displaying the containing line.

This is especially valuable when a single line contains nested indexing, calls, arithmetic, or attribute access.

In [2]:
def calculate_invoice(order: dict[str, object]) -> float:
    # Deliberately dense to demonstrate precise expression positions.
    return round(order['subtotal'] * (1 + order['tax']['rate']), 2)

broken_order = {
    'subtotal': 120.0,
    'tax': None,
}

try:
    calculate_invoice(broken_order)
except Exception:
    traceback.print_exc(limit=2)

Traceback (most recent call last):
  File "C:\Users\user1\AppData\Local\Temp\ipykernel_13052\3092855701.py", line 11, in <module>
    calculate_invoice(broken_order)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "C:\Users\user1\AppData\Local\Temp\ipykernel_13052\3092855701.py", line 3, in calculate_invoice
    return round(order['subtotal'] * (1 + order['tax']['rate']), 2)
                                          ~~~~~~~~~~~~^^^^^^^^
TypeError: 'NoneType' object is not subscriptable


The exact caret rendering depends on the frontend, but Python 3.11 stores start and end columns for many expressions. Those positions are also available programmatically.

In [3]:
def inspect_syntax_error(source: str) -> dict[str, object]:
    try:
        compile(source, '<exercise>', 'exec')
    except SyntaxError as exc:
        return {
            'message': exc.msg,
            'line': exc.lineno,
            'offset': exc.offset,
            'end_line': exc.end_lineno,
            'end_offset': exc.end_offset,
            'text': exc.text,
        }
    raise AssertionError('Expected invalid syntax')

syntax_details = inspect_syntax_error("result = [x * 2 for x range(5)]")
syntax_details

{'message': "'in' expected after for-loop variables",
 'line': 1,
 'offset': 23,
 'end_line': 1,
 'end_offset': 28,
 'text': 'result = [x * 2 for x range(5)]\n'}

## Problem 1 — Build a precise syntax-error reporter

Write `format_syntax_error(source)` so that it:

1. compiles `source` in `exec` mode;
2. returns `"valid"` when compilation succeeds;
3. otherwise returns a three-line string containing the source line, a caret span, and the error message;
4. uses both `offset` and `end_offset` when available;
5. never crashes when the parser omits an end position.

### Solution 1

In [4]:
def format_syntax_error(source: str) -> str:
    try:
        compile(source, '<input>', 'exec')
    except SyntaxError as exc:
        line = (exc.text or '').rstrip('\n')
        start = max((exc.offset or 1) - 1, 0)
        end_offset = exc.end_offset if exc.end_offset is not None else (exc.offset or 1)
        width = max(end_offset - (exc.offset or 1), 1)
        marker = ' ' * start + '^' * width
        return f'{line}\n{marker}\nSyntaxError: {exc.msg}'
    else:
        return 'valid'

report = format_syntax_error("items = {'a': 1 'b': 2}")
print(report)
assert '^' in report
assert 'SyntaxError:' in report
assert format_syntax_error('x = 10') == 'valid'

items = {'a': 1 'b': 2}
              ^^^^^
SyntaxError: invalid syntax. Perhaps you forgot a comma?


### Best-practice takeaway

Do not manually parse traceback text when structured attributes such as `lineno`, `offset`, `end_lineno`, and `end_offset` are available. Structured data is more stable and easier to test.

# 2. PEP 678 — Enriching exceptions with notes

`BaseException.add_note()` appends contextual information without replacing the original exception message or losing its type.

Useful notes include:

- record identifiers,
- retry numbers,
- configuration paths,
- user-facing remediation hints,
- operation names in batch processing.

In [5]:
def parse_positive_int(raw: str, *, field: str) -> int:
    try:
        value = int(raw)
        if value <= 0:
            raise ValueError('value must be positive')
        return value
    except ValueError as exc:
        exc.add_note(f'field={field!r}')
        exc.add_note(f'raw_value={raw!r}')
        raise

try:
    parse_positive_int('-8', field='worker_count')
except ValueError as exc:
    print('message:', exc)
    print('notes:', exc.__notes__)

message: value must be positive
notes: ["field='worker_count'", "raw_value='-8'"]


## Problem 2 — Preserve root cause while adding batch context

Implement `convert_records(records)`.

Each record contains `{"id": ..., "quantity": ...}`. Convert `quantity` to a positive integer. For every invalid record:

- keep the original exception type;
- add notes containing the record index and ID;
- continue processing later records;
- return `(converted, errors)`.

### Solution 2

In [6]:
def convert_records(
    records: list[dict[str, object]],
) -> tuple[list[dict[str, object]], list[Exception]]:
    converted: list[dict[str, object]] = []
    errors: list[Exception] = []

    for index, record in enumerate(records):
        try:
            quantity = int(record['quantity'])
            if quantity <= 0:
                raise ValueError('quantity must be positive')
            converted.append({**record, 'quantity': quantity})
        except (KeyError, TypeError, ValueError) as exc:
            exc.add_note(f'record_index={index}')
            exc.add_note(f"record_id={record.get('id', '<missing>')!r}")
            errors.append(exc)

    return converted, errors

records = [
    {'id': 'A', 'quantity': '3'},
    {'id': 'B', 'quantity': 'zero'},
    {'id': 'C'},
    {'id': 'D', 'quantity': -2},
]

converted, conversion_errors = convert_records(records)
print(converted)
for error in conversion_errors:
    print(type(error).__name__, str(error), error.__notes__)

assert converted == [{'id': 'A', 'quantity': 3}]
assert len(conversion_errors) == 3
assert all(error.__notes__ for error in conversion_errors)

[{'id': 'A', 'quantity': 3}]
ValueError invalid literal for int() with base 10: 'zero' ['record_index=1', "record_id='B'"]
KeyError 'quantity' ['record_index=2', "record_id='C'"]
ValueError quantity must be positive ['record_index=3', "record_id='D'"]


### Best-practice takeaway

Use notes for context that is discovered while an exception propagates. Avoid rewriting every exception as a generic `RuntimeError`, because doing so can make selective handling and diagnosis harder.

# 3. PEP 654 — `ExceptionGroup` and `except*`

An `ExceptionGroup` represents multiple unrelated failures. `except* SomeError` extracts every matching leaf exception into a subgroup. Unmatched leaves continue propagating automatically.

Important rules:

- `except*` handlers receive a group, even when only one leaf matches;
- one `except*` clause runs at most once for its matching subgroup;
- ordinary `except` and `except*` cannot be mixed in the same `try` statement;
- raising an exception group from an existing API can be an API-breaking change.

In [7]:
errors = ExceptionGroup(
    'import failures',
    [
        ValueError('invalid price'),
        OSError('inventory file unavailable'),
        ExceptionGroup(
            'nested validation failures',
            [TypeError('sku must be text'), ValueError('quantity must be positive')],
        ),
    ],
)

try:
    raise errors
except* ValueError as value_group:
    print('Value-related subgroup:')
    traceback.print_exception(value_group)
except* OSError as os_group:
    print('OS-related subgroup:')
    traceback.print_exception(os_group)
except* TypeError as type_group:
    print('Type-related subgroup:')
    traceback.print_exception(type_group)

Value-related subgroup:
OS-related subgroup:
Type-related subgroup:


  + Exception Group Traceback (most recent call last):
  |   File "C:\Users\user1\AppData\Local\Temp\ipykernel_13052\982458917.py", line 14, in <module>
  |     raise errors
  | ExceptionGroup: import failures (2 sub-exceptions)
  +-+---------------- 1 ----------------
    | ValueError: invalid price
    +---------------- 2 ----------------
    | ExceptionGroup: nested validation failures (1 sub-exception)
    +-+---------------- 1 ----------------
      | ValueError: quantity must be positive
      +------------------------------------
  + Exception Group Traceback (most recent call last):
  |   File "C:\Users\user1\AppData\Local\Temp\ipykernel_13052\982458917.py", line 14, in <module>
  |     raise errors
  | ExceptionGroup: import failures (1 sub-exception)
  +-+---------------- 1 ----------------
    | OSError: inventory file unavailable
    +------------------------------------
  + Exception Group Traceback (most recent call last):
  |   File "C:\Users\user1\AppData\Local\Temp\ipy

In [8]:
def iter_leaf_exceptions(exc: BaseException):
    if isinstance(exc, BaseExceptionGroup):
        for child in exc.exceptions:
            yield from iter_leaf_exceptions(child)
    else:
        yield exc

leaf_types = [type(exc).__name__ for exc in iter_leaf_exceptions(errors)]
leaf_types

['ValueError', 'OSError', 'TypeError', 'ValueError']

## Problem 3 — Validate an entire payload and report all failures

Implement `validate_user_payload(payload)` with these rules:

- `name` must be a non-empty string;
- `age` must be an integer from 18 through 120;
- `email` must contain exactly one `@`;
- `roles` must be a non-empty list of strings.

Collect every failure and raise one `ExceptionGroup`. Add a note identifying the failing field to each leaf exception.

### Solution 3

In [9]:
def validate_user_payload(payload: dict[str, object]) -> dict[str, object]:
    failures: list[Exception] = []

    def capture(field: str, check) -> None:
        try:
            check()
        except (KeyError, TypeError, ValueError) as exc:
            exc.add_note(f'field={field!r}')
            failures.append(exc)

    def check_name() -> None:
        name = payload['name']
        if not isinstance(name, str):
            raise TypeError('name must be a string')
        if not name.strip():
            raise ValueError('name must not be blank')

    def check_age() -> None:
        age = payload['age']
        if type(age) is not int:  # reject bool, which is a subclass of int
            raise TypeError('age must be an integer')
        if not 18 <= age <= 120:
            raise ValueError('age must be between 18 and 120')

    def check_email() -> None:
        email = payload['email']
        if not isinstance(email, str):
            raise TypeError('email must be a string')
        if email.count('@') != 1:
            raise ValueError('email must contain exactly one @')

    def check_roles() -> None:
        roles = payload['roles']
        if not isinstance(roles, list):
            raise TypeError('roles must be a list')
        if not roles:
            raise ValueError('roles must not be empty')
        if not all(isinstance(role, str) for role in roles):
            raise TypeError('every role must be a string')

    capture('name', check_name)
    capture('age', check_age)
    capture('email', check_email)
    capture('roles', check_roles)

    if failures:
        raise ExceptionGroup('user payload validation failed', failures)

    return payload

bad_payload = {
    'name': '   ',
    'age': True,
    'email': 'invalid@@example.com',
    'roles': [],
}

handled_fields: list[str] = []
try:
    validate_user_payload(bad_payload)
except* TypeError as type_errors:
    for leaf in iter_leaf_exceptions(type_errors):
        handled_fields.extend(leaf.__notes__)
        print('TYPE:', leaf, leaf.__notes__)
except* ValueError as value_errors:
    for leaf in iter_leaf_exceptions(value_errors):
        handled_fields.extend(leaf.__notes__)
        print('VALUE:', leaf, leaf.__notes__)

assert len(handled_fields) == 4

TYPE: age must be an integer ["field='age'"]
VALUE: name must not be blank ["field='name'"]
VALUE: email must contain exactly one @ ["field='email'"]
VALUE: roles must not be empty ["field='roles'"]


## Problem 4 — Selectively split an exception group

Given a nested group, separate retryable failures (`TimeoutError` and `ConnectionError`) from permanent failures without flattening the original hierarchy.

### Solution 4

In [10]:
service_failures = ExceptionGroup(
    'service failures',
    [
        TimeoutError('catalog timeout'),
        ValueError('invalid product payload'),
        ExceptionGroup(
            'regional failures',
            [ConnectionError('eu-west disconnected'), PermissionError('access denied')],
        ),
    ],
)

retryable, permanent = service_failures.split((TimeoutError, ConnectionError))

print('Retryable leaves:')
for leaf in iter_leaf_exceptions(retryable):
    print(' -', type(leaf).__name__, leaf)

print('Permanent leaves:')
for leaf in iter_leaf_exceptions(permanent):
    print(' -', type(leaf).__name__, leaf)

assert {type(x) for x in iter_leaf_exceptions(retryable)} == {TimeoutError, ConnectionError}
assert {type(x) for x in iter_leaf_exceptions(permanent)} == {ValueError, PermissionError}

Retryable leaves:
 - TimeoutError catalog timeout
 - ConnectionError eu-west disconnected
Permanent leaves:
 - ValueError invalid product payload
 - PermissionError access denied


# 4. Structured concurrency with `asyncio.TaskGroup`

`asyncio.TaskGroup` is an asynchronous context manager for related child tasks.

On successful exit, every task has completed. If one task fails with a non-cancellation exception, sibling tasks are cancelled and the failures are raised as an exception group.

This makes task lifetime explicit and avoids leaking un-awaited tasks.

In [11]:
async def delayed_square(value: int, delay: float) -> int:
    await asyncio.sleep(delay)
    return value * value

async def successful_task_group() -> list[int]:
    async with asyncio.TaskGroup() as group:
        tasks = [
            group.create_task(delayed_square(value, delay=0.005 * value))
            for value in range(1, 5)
        ]
    return [task.result() for task in tasks]

successful_results = await successful_task_group()
print(successful_results)
assert successful_results == [1, 4, 9, 16]

[1, 4, 9, 16]


In [12]:
async def timeout_demo() -> str:
    try:
        async with asyncio.timeout(0.01):
            await asyncio.sleep(0.05)
    except TimeoutError:
        return 'timed out as expected'
    return 'unexpectedly completed'

await timeout_demo()

'timed out as expected'

## Problem 5 — Run independent validators concurrently

Create three async validators for a username:

- a length validator that raises `ValueError`;
- a reserved-name validator that raises `PermissionError`;
- a simulated uniqueness validator that raises `LookupError`.

Use a synchronization gate so all failing validators can raise before cancellation suppresses siblings. Handle each exception category with `except*` and return a list of readable messages.

### Solution 5

In [13]:
async def validate_length(name: str, gate: asyncio.Event) -> None:
    await gate.wait()
    if len(name) < 5:
        raise ValueError('username must contain at least five characters')

async def validate_reserved(name: str, gate: asyncio.Event) -> None:
    await gate.wait()
    if name.lower() in {'root', 'admin', 'sys'}:
        raise PermissionError('username is reserved')

async def validate_unique(name: str, gate: asyncio.Event) -> None:
    await gate.wait()
    if name.lower() in {'root', 'alice'}:
        raise LookupError('username is already registered')

async def validate_username_concurrently(name: str) -> list[str]:
    gate = asyncio.Event()
    messages: list[str] = []

    try:
        async with asyncio.TaskGroup() as group:
            group.create_task(validate_length(name, gate))
            group.create_task(validate_reserved(name, gate))
            group.create_task(validate_unique(name, gate))
            await asyncio.sleep(0)  # allow all tasks to begin waiting
            gate.set()
    except* ValueError as group:
        messages.extend(str(exc) for exc in iter_leaf_exceptions(group))
    except* PermissionError as group:
        messages.extend(str(exc) for exc in iter_leaf_exceptions(group))
    except* LookupError as group:
        messages.extend(str(exc) for exc in iter_leaf_exceptions(group))

    return sorted(messages)

username_messages = await validate_username_concurrently('root')
print(username_messages)
assert len(username_messages) == 3

['username is already registered', 'username is reserved', 'username must contain at least five characters']


## Problem 6 — Bounded structured-concurrency map

Implement `bounded_map(func, values, limit, timeout)` using:

- `asyncio.Semaphore` to cap concurrency;
- `asyncio.TaskGroup` to own every task;
- `asyncio.timeout` to bound the entire operation;
- result ordering identical to input ordering.

### Solution 6

In [14]:
async def bounded_map(func, values, *, limit: int, timeout: float):
    if limit < 1:
        raise ValueError('limit must be at least 1')

    semaphore = asyncio.Semaphore(limit)
    results = [None] * len(values)

    async def run_one(index: int, value) -> None:
        async with semaphore:
            results[index] = await func(value)

    async with asyncio.timeout(timeout):
        async with asyncio.TaskGroup() as group:
            for index, value in enumerate(values):
                group.create_task(run_one(index, value))

    return results

active = 0
peak_active = 0

async def instrumented_double(value: int) -> int:
    global active, peak_active
    active += 1
    peak_active = max(peak_active, active)
    try:
        await asyncio.sleep(0.005)
        return value * 2
    finally:
        active -= 1

mapped = await bounded_map(instrumented_double, list(range(10)), limit=3, timeout=1.0)
print(mapped, peak_active)
assert mapped == [value * 2 for value in range(10)]
assert peak_active <= 3

[0, 2, 4, 6, 8, 10, 12, 14, 16, 18] 3


### Best-practice takeaway

Use `TaskGroup` when tasks form one logical operation. Use independent background-task management only when task lifetime truly extends beyond the current scope.

# 5. `tomllib` — TOML parsing in the standard library

Python 3.11 adds the read-only `tomllib` module.

- `tomllib.loads()` parses a string.
- `tomllib.load()` reads from a binary file object.
- `parse_float=` can preserve decimal precision.
- Writing TOML is intentionally outside the module's scope.

In [15]:
TOML_TEXT = r"""
[service]
name = "billing"
workers = 4
timeout_seconds = 2.5

[database]
host = "db.internal"
ports = [5432, 5433]

[features]
audit = true
experimental = false
"""

config = tomllib.loads(TOML_TEXT)
config

{'service': {'name': 'billing', 'workers': 4, 'timeout_seconds': 2.5},
 'database': {'host': 'db.internal', 'ports': [5432, 5433]},
 'features': {'audit': True, 'experimental': False}}

In [16]:
precise = tomllib.loads('tax_rate = 0.075', parse_float=Decimal)
print(precise, type(precise['tax_rate']))
assert precise['tax_rate'] == Decimal('0.075')

{'tax_rate': Decimal('0.075')} <class 'decimal.Decimal'>


In [17]:
binary_config = BytesIO(TOML_TEXT.encode('utf-8'))
loaded_from_binary = tomllib.load(binary_config)
assert loaded_from_binary == config

## Problem 7 — Parse and validate a service configuration

Build `load_service_config(text)` that returns a frozen dataclass with:

- `name: str`;
- `workers: int` in the range 1–64;
- `timeout_seconds: Decimal` greater than zero;
- `audit: bool`.

Collect all validation failures in an `ExceptionGroup`. Add a note with the TOML key path to each error.

### Solution 7

In [18]:
@dataclass(frozen=True, slots=True)
class ServiceConfig:
    name: str
    workers: int
    timeout_seconds: Decimal
    audit: bool


def load_service_config(text: str) -> ServiceConfig:
    parsed = tomllib.loads(text, parse_float=Decimal)
    failures: list[Exception] = []

    def read(path: tuple[str, ...], expected_type):
        current = parsed
        try:
            for part in path:
                current = current[part]
            if not isinstance(current, expected_type):
                raise TypeError(
                    f'expected {expected_type.__name__}, got {type(current).__name__}'
                )
            return current
        except (KeyError, TypeError) as exc:
            exc.add_note('toml_path=' + '.'.join(path))
            failures.append(exc)
            return None

    name = read(('service', 'name'), str)
    workers = read(('service', 'workers'), int)
    timeout_seconds = read(('service', 'timeout_seconds'), Decimal)
    audit = read(('features', 'audit'), bool)

    if isinstance(name, str) and not name.strip():
        exc = ValueError('service name must not be blank')
        exc.add_note('toml_path=service.name')
        failures.append(exc)

    if type(workers) is int and not 1 <= workers <= 64:
        exc = ValueError('workers must be between 1 and 64')
        exc.add_note('toml_path=service.workers')
        failures.append(exc)

    if isinstance(timeout_seconds, Decimal) and timeout_seconds <= 0:
        exc = ValueError('timeout_seconds must be positive')
        exc.add_note('toml_path=service.timeout_seconds')
        failures.append(exc)

    if failures:
        raise ExceptionGroup('invalid service configuration', failures)

    return ServiceConfig(
        name=name,
        workers=workers,
        timeout_seconds=timeout_seconds,
        audit=audit,
    )

valid_config = load_service_config(TOML_TEXT)
print(valid_config)
assert valid_config.workers == 4
assert valid_config.timeout_seconds == Decimal('2.5')

ServiceConfig(name='billing', workers=4, timeout_seconds=Decimal('2.5'), audit=True)


In [19]:
INVALID_TOML_CONFIG = r"""
[service]
name = ""
workers = 0
timeout_seconds = -1.25

[features]
audit = "yes"
"""

config_error_count = 0
try:
    load_service_config(INVALID_TOML_CONFIG)
except* (TypeError, ValueError) as group:
    leaves = list(iter_leaf_exceptions(group))
    config_error_count = len(leaves)
    for leaf in leaves:
        print(type(leaf).__name__, leaf, leaf.__notes__)

assert config_error_count == 4

TypeError expected bool, got str ['toml_path=features.audit']
ValueError service name must not be blank ['toml_path=service.name']
ValueError workers must be between 1 and 64 ['toml_path=service.workers']
ValueError timeout_seconds must be positive ['toml_path=service.timeout_seconds']


# 6. Typing improvements in Python 3.11

The following tools improve static contracts. They generally do not enforce correctness at runtime by themselves.

- `Self`: methods returning the most precise current class type;
- `Required` / `NotRequired`: per-key `TypedDict` presence rules;
- `LiteralString`: marks strings derived only from literals for security-sensitive APIs;
- `Never` and `assert_never`: exhaustive control-flow checking;
- `TypeVarTuple` and `Unpack`: variadic generics.

## 6.1 `Self` for fluent and subclass-preserving APIs

In [20]:
@dataclass(frozen=True)
class Query:
    clauses: tuple[str, ...] = ()

    def where(self, clause: str) -> Self:
        return type(self)(self.clauses + (clause,))

    @classmethod
    def empty(cls) -> Self:
        return cls()

@dataclass(frozen=True)
class AuditedQuery(Query):
    pass

query = AuditedQuery.empty().where('active = true').where('age >= 18')
print(query)
assert type(query) is AuditedQuery

AuditedQuery(clauses=('active = true', 'age >= 18'))


## Problem 8 — Immutable vector API with `Self`

Implement an immutable `Vector2D` with methods `scaled()` and `translated()` that preserve subclasses. Demonstrate that a `NamedVector` remains a `NamedVector` after chaining.

### Solution 8

In [21]:
@dataclass(frozen=True)
class Vector2D:
    x: float
    y: float

    def scaled(self, factor: float) -> Self:
        return type(self)(self.x * factor, self.y * factor)

    def translated(self, dx: float, dy: float) -> Self:
        return type(self)(self.x + dx, self.y + dy)

@dataclass(frozen=True)
class NamedVector(Vector2D):
    # A default keeps the inherited two-argument construction compatible.
    name: str = 'vector'

    def scaled(self, factor: float) -> Self:
        return type(self)(self.x * factor, self.y * factor, self.name)

    def translated(self, dx: float, dy: float) -> Self:
        return type(self)(self.x + dx, self.y + dy, self.name)

v = NamedVector(1.0, 2.0, 'velocity').scaled(3).translated(-1, 4)
print(v)
assert isinstance(v, NamedVector)
assert v == NamedVector(2.0, 10.0, 'velocity')

NamedVector(x=2.0, y=10.0, name='velocity')


## 6.2 `Required` and `NotRequired` in `TypedDict`

In [22]:
class UserPatch(TypedDict, total=False):
    user_id: Required[int]
    display_name: str
    email: str
    reason: NotRequired[str]

print('required:', UserPatch.__required_keys__)
print('optional:', UserPatch.__optional_keys__)
assert UserPatch.__required_keys__ == frozenset({'user_id'})

required: frozenset({'user_id'})
optional: frozenset({'email', 'reason', 'display_name'})


## Problem 9 — Runtime validator driven by `TypedDict` metadata

Write a small runtime validator that checks required-key presence using `__required_keys__`, while clearly documenting that full value-type checking still belongs to a dedicated validator or static checker.

### Solution 9

In [23]:
def require_typed_dict_keys(schema: type, value: dict[str, object]) -> None:
    missing = schema.__required_keys__ - value.keys()
    if missing:
        raise KeyError(f'missing required keys: {sorted(missing)}')

require_typed_dict_keys(UserPatch, {'user_id': 42, 'email': 'a@example.com'})

try:
    require_typed_dict_keys(UserPatch, {'email': 'a@example.com'})
except KeyError as exc:
    print(exc)
else:
    raise AssertionError('Expected a missing-key failure')

"missing required keys: ['user_id']"


## 6.3 `LiteralString` for security-sensitive string APIs

In [24]:
def execute_static_sql(sql: LiteralString) -> None:
    """A static type checker can reject arbitrary runtime strings here."""
    print('would execute:', sql)

SAFE_QUERY: LiteralString = 'SELECT id, name FROM users WHERE active = 1'
execute_static_sql(SAFE_QUERY)

print(get_type_hints(execute_static_sql))

would execute: SELECT id, name FROM users WHERE active = 1
{'sql': typing.LiteralString, 'return': <class 'NoneType'>}


`LiteralString` is not a runtime SQL-injection defense. Parameterized SQL remains the correct runtime design. The annotation helps a static checker flag dangerous string construction at API boundaries.

## 6.4 Exhaustiveness with `Never` and `assert_never`

In [25]:
class JobState(StrEnum):
    QUEUED = auto()
    RUNNING = auto()
    SUCCEEDED = auto()
    FAILED = auto()


def terminal_message(state: JobState) -> str:
    match state:
        case JobState.SUCCEEDED:
            return 'completed successfully'
        case JobState.FAILED:
            return 'completed with errors'
        case JobState.QUEUED | JobState.RUNNING:
            return 'not terminal'
        case _ as unreachable:
            assert_never(unreachable)

assert terminal_message(JobState.SUCCEEDED) == 'completed successfully'

## 6.5 Variadic generics with `TypeVarTuple`

In [26]:
Ts = TypeVarTuple('Ts')


def pack(*values: Unpack[Ts]) -> tuple[Unpack[Ts]]:
    return values

packed = pack(1, 'two', 3.0)
print(packed)
assert packed == (1, 'two', 3.0)

(1, 'two', 3.0)


## Problem 10 — Heterogeneous row wrapper

Create a generic `Row[*Ts]`-style wrapper using the Python 3.11-compatible `Generic[Unpack[Ts]]` spelling. Store and return a heterogeneous tuple without erasing the element types for static checkers.

### Solution 10

In [27]:
RowTypes = TypeVarTuple('RowTypes')

@dataclass(frozen=True)
class Row(Generic[Unpack[RowTypes]]):
    values: tuple[Unpack[RowTypes]]

    def as_tuple(self) -> tuple[Unpack[RowTypes]]:
        return self.values

row = Row((101, 'Ada', True, Decimal('9.5')))
print(row.as_tuple())
assert row.as_tuple()[1] == 'Ada'

(101, 'Ada', True, Decimal('9.5'))


# 7. `StrEnum` — string constants with enum semantics

`StrEnum` members are also strings. With `auto()`, values become the lower-cased member name.

This is useful for protocol values, JSON fields, environment names, state machines, and command names.

In [28]:
class Environment(StrEnum):
    DEVELOPMENT = auto()
    STAGING = auto()
    PRODUCTION = auto()

print(Environment.DEVELOPMENT)
print(Environment.DEVELOPMENT.value)
print(json.dumps({'environment': Environment.PRODUCTION}))

assert Environment.DEVELOPMENT == 'development'
assert Environment.PRODUCTION.value == 'production'

development
development
{"environment": "production"}


## Problem 11 — Robust command parsing with `StrEnum`

Implement a command enum and parser that:

- accepts surrounding whitespace and mixed case;
- returns a `Command` member;
- raises a `ValueError` with an allowed-values note for unknown input.

### Solution 11

In [29]:
class Command(StrEnum):
    START = auto()
    STOP = auto()
    STATUS = auto()
    RESTART = auto()


def parse_command(raw: str) -> Command:
    normalized = raw.strip().lower()
    try:
        return Command(normalized)
    except ValueError as exc:
        allowed = ', '.join(command.value for command in Command)
        exc.add_note(f'allowed_commands={allowed}')
        raise

assert parse_command('  ReStArT ') is Command.RESTART

try:
    parse_command('delete')
except ValueError as exc:
    print(exc, exc.__notes__)

'delete' is not a valid Command ['allowed_commands=start, stop, status, restart']


### Compatibility note

Some older APIs use exact checks such as `type(value) is str` instead of `isinstance(value, str)`. In those cases pass `member.value` or `str(member)` explicitly.

# 8. Regular expressions: atomic groups and possessive quantifiers

Python 3.11 adds:

- atomic groups: `(?>...)`;
- possessive quantifiers: `*+`, `++`, `?+`, and `{m,n}+`.

They prevent the regex engine from backtracking into a matched region. This can improve performance and can intentionally change matching semantics.

In [30]:
print(re.fullmatch(r'a*a', 'aaaa') is not None)
print(re.fullmatch(r'a*+a', 'aaaa') is not None)

# Greedy a* backtracks to leave one 'a' for the final token.
assert re.fullmatch(r'a*a', 'aaaa') is not None
# Possessive a*+ refuses to give characters back.
assert re.fullmatch(r'a*+a', 'aaaa') is None

True
False


In [31]:
catastrophic = re.compile(r'(a+)+$')
possessive = re.compile(r'(a++)+$')
subject = 'a' * 18 + '!'

slow_time = timeit.timeit(lambda: catastrophic.fullmatch(subject), number=1)
fast_time = timeit.timeit(lambda: possessive.fullmatch(subject), number=1)
print({'backtracking_seconds': slow_time, 'possessive_seconds': fast_time})

{'backtracking_seconds': 0.01582029927521944, 'possessive_seconds': 7.200054824352264e-06}


## Problem 12 — Validate dotted identifiers without unnecessary backtracking

Accept strings such as `service.database.host` where each segment:

- starts with a letter or underscore;
- continues with letters, digits, or underscores;
- is separated by exactly one dot.

Use an atomic group or possessive quantifier in the implementation, and test valid and invalid cases.

### Solution 12

In [32]:
DOTTED_IDENTIFIER = re.compile(
    r'^(?>[A-Za-z_][A-Za-z0-9_]*)(?:\.(?>[A-Za-z_][A-Za-z0-9_]*))*$'
)

valid_identifiers = [
    'service',
    'service.database.host',
    '_private.value2',
]
invalid_identifiers = [
    '',
    '2service',
    'service..host',
    'service.-host',
    'service.',
]

assert all(DOTTED_IDENTIFIER.fullmatch(value) for value in valid_identifiers)
assert all(not DOTTED_IDENTIFIER.fullmatch(value) for value in invalid_identifiers)
print('identifier tests passed')

identifier tests passed


### Best-practice takeaway

Do not add atomicity blindly. It removes possible matches as well as backtracking. First define the intended language, then use atomic or possessive constructs where giving characters back can never produce a valid alternative.

# 9. Additional useful Python 3.11 changes

## 9.1 Starred expressions in `for` iterable expressions

Python 3.11 allows unpacking directly in the iterable expression of a `for` loop.

In [33]:
primary = ('alpha', 'beta')
secondary = ('gamma', 'delta')

combined = []
for item in *primary, *secondary:
    combined.append(item.upper())

print(combined)
assert combined == ['ALPHA', 'BETA', 'GAMMA', 'DELTA']

['ALPHA', 'BETA', 'GAMMA', 'DELTA']


## 9.2 PEP 682 — signed-zero formatting

The `z` formatting option normalizes negative zero after rounding. This is useful in reports where `-0.00` would be distracting or misleading.

In [34]:
values = [-0.004, -0.0001, 0.0, 1.234]
normal = [f'{value:.2f}' for value in values]
normalized = [f'{value:z.2f}' for value in values]

print('normal:    ', normal)
print('normalized:', normalized)
assert normalized[0] == '0.00'
assert normalized[1] == '0.00'

normal:     ['-0.00', '-0.00', '0.00', '1.23']
normalized: ['0.00', '0.00', '0.00', '1.23']


## Problem 13 — Financial variance formatter

Format a decimal variance with:

- two decimal places;
- a leading sign for non-zero values;
- no `-0.00` after rounding;
- thousands separators.

### Solution 13

In [35]:
def format_variance(value: float) -> str:
    return f'{value:+z,.2f}'

assert format_variance(-0.001) == '+0.00'
assert format_variance(12345.678) == '+12,345.68'
assert format_variance(-12345.678) == '-12,345.68'
print([format_variance(v) for v in (-0.001, 0, 12345.678, -12345.678)])

['+0.00', '+0.00', '+12,345.68', '-12,345.68']


## 9.3 Integer-string conversion length limit

Python 3.11 limits decimal integer-to-string and string-to-integer conversions to reduce denial-of-service risk from extremely large decimal strings.

The exact configured limit is available through `sys.get_int_max_str_digits()`. Power-of-two bases such as hexadecimal are not subject to the same algorithmic limit.

In [36]:
current_limit = sys.get_int_max_str_digits()
print('configured decimal digit limit:', current_limit)
assert current_limit == 0 or current_limit >= 640

configured decimal digit limit: 4300


## Problem 14 — Defensive decimal integer parser

Implement `parse_bounded_decimal(raw, max_digits)` that:

- accepts optional leading `+` or `-`;
- rejects non-decimal characters;
- counts digits before calling `int()`;
- rejects input longer than the application limit;
- adds a note containing the configured interpreter limit when `int()` itself raises.

### Solution 14

In [37]:
def parse_bounded_decimal(raw: str, *, max_digits: int = 1000) -> int:
    if max_digits < 1:
        raise ValueError('max_digits must be positive')

    text = raw.strip()
    digits = text[1:] if text[:1] in {'+', '-'} else text

    if not digits or not digits.isdecimal():
        raise ValueError('expected a base-10 integer string')
    if len(digits) > max_digits:
        raise ValueError(f'decimal integer exceeds application limit of {max_digits} digits')

    try:
        return int(text)
    except ValueError as exc:
        exc.add_note(f'interpreter_digit_limit={sys.get_int_max_str_digits()}')
        raise

assert parse_bounded_decimal('-12345', max_digits=5) == -12345

try:
    parse_bounded_decimal('9' * 101, max_digits=100)
except ValueError as exc:
    print(exc)
else:
    raise AssertionError('Expected an application-limit failure')

decimal integer exceeds application limit of 100 digits


## 9.4 `datetime.UTC`

Python 3.11 adds `datetime.UTC` as a convenient alias for the UTC singleton.

In [38]:
from datetime import UTC, datetime

now_utc = datetime.now(UTC)
print(now_utc.isoformat())
assert now_utc.tzinfo is UTC

2026-09-03T11:20:54.518040+00:00


# 10. Faster CPython and the specializing adaptive interpreter

Python 3.11 introduced major interpreter optimizations. CPython can specialize bytecode instructions based on observed runtime types.

Performance lessons:

- measure representative workloads rather than relying on headline averages;
- warm up functions before inspecting adaptive specialization;
- do not write code against private cache layout or specific opcodes;
- use `dis` for learning and diagnosis, not as a stable application API contract.

In [39]:
def total_prices(prices: list[float]) -> float:
    total = 0.0
    for price in prices:
        total += price
    return total

sample_prices = [float(i) for i in range(100)]
for _ in range(20_000):
    total_prices(sample_prices)

# In CPython 3.11+, adaptive=True exposes specialized instructions when available.
dis.dis(total_prices, adaptive=True, show_caches=True)

  1           RESUME_CHECK             0

  2           LOAD_CONST               1 (0.0)
              STORE_FAST               1 (total)

  3           LOAD_FAST                0 (prices)
              GET_ITER
      L1:     FOR_ITER_LIST            7 (to L2)
              CACHE                    0 (counter: 832)
              STORE_FAST               2 (price)

  4           LOAD_FAST_LOAD_FAST     18 (total, price)
              BINARY_OP_ADD_FLOAT     13 (+=)
              CACHE                    0 (counter: 832)
              STORE_FAST               1 (total)
              JUMP_BACKWARD            9 (to L1)
              CACHE                    0 (counter: 260)

  3   L2:     END_FOR
              POP_TOP

  5           LOAD_FAST                1 (total)
              RETURN_VALUE


## Problem 15 — Compare monomorphic and polymorphic call sites

Create two functions:

- one repeatedly adds integers;
- one repeatedly applies `+` to alternating compatible types in separate calls.

Warm them up, inspect their disassembly, and benchmark them. Do not assert an exact speed ratio because hardware, build flags, Python patch version, and workload all matter.

### Solution 15

In [40]:
def add_many_ints(values: list[int]) -> int:
    total = 0
    for value in values:
        total += value
    return total


def concatenate_many(values: list[str]) -> str:
    total = ''
    for value in values:
        total += value
    return total

ints = list(range(100))
strings = ['x'] * 100

for _ in range(10_000):
    add_many_ints(ints)
    concatenate_many(strings)

int_time = timeit.timeit(lambda: add_many_ints(ints), number=5_000)
str_time = timeit.timeit(lambda: concatenate_many(strings), number=5_000)
print({'integer_addition_seconds': int_time, 'string_concatenation_seconds': str_time})

print('\nInteger function:')
dis.dis(add_many_ints, adaptive=True, show_caches=True)
print('\nString function:')
dis.dis(concatenate_many, adaptive=True, show_caches=True)

assert add_many_ints([1, 2, 3]) == 6
assert concatenate_many(['a', 'b', 'c']) == 'abc'

{'integer_addition_seconds': 0.023348000831902027, 'string_concatenation_seconds': 0.053437200374901295}

Integer function:
  1           RESUME_CHECK             0

  2           LOAD_CONST               1 (0)
              STORE_FAST               1 (total)

  3           LOAD_FAST                0 (values)
              GET_ITER
      L1:     FOR_ITER_LIST            7 (to L2)
              CACHE                    0 (counter: 832)
              STORE_FAST               2 (value)

  4           LOAD_FAST_LOAD_FAST     18 (total, value)
              BINARY_OP_ADD_INT       13 (+=)
              CACHE                    0 (counter: 832)
              STORE_FAST               1 (total)
              JUMP_BACKWARD            9 (to L1)
              CACHE                    0 (counter: 260)

  3   L2:     END_FOR
              POP_TOP

  5           LOAD_FAST                1 (total)
              RETURN_VALUE

String function:
  8           RESUME_CHECK             0

  9           LOA

# 11. Capstone — Concurrent TOML-driven job runner

This capstone combines:

- `tomllib` configuration parsing;
- `StrEnum` for operation names;
- `TypedDict` required/optional fields;
- a frozen dataclass model;
- `asyncio.TaskGroup` and `asyncio.timeout`;
- `ExceptionGroup` and `except*`;
- exception notes;
- signed-zero formatting;
- exhaustive matching with `assert_never`.

## Capstone problem

Parse a TOML document containing jobs. Each job has:

- `id`;
- `operation` (`add`, `divide`, or `sqrt`);
- `left`;
- optional `right`;
- optional `delay_ms`.

Requirements:

1. Validate all jobs before execution and report every configuration failure together.
2. Run valid jobs concurrently with one overall timeout.
3. Preserve output order.
4. Attach job IDs as notes to execution failures.
5. Handle arithmetic errors separately from timeout errors.
6. Format successful numeric results with signed-zero normalization.

### Capstone solution

In [41]:
class Operation(StrEnum):
    ADD = auto()
    DIVIDE = auto()
    SQRT = auto()


class RawJob(TypedDict):
    id: Required[str]
    operation: Required[str]
    left: Required[float]
    right: NotRequired[float]
    delay_ms: NotRequired[int]


@dataclass(frozen=True, slots=True)
class Job:
    id: str
    operation: Operation
    left: float
    right: float | None = None
    delay_ms: int = 0


@dataclass(frozen=True, slots=True)
class JobResult:
    id: str
    value: float

    def formatted(self) -> str:
        return f'{self.value:z.3f}'

In [42]:
def parse_jobs(text: str) -> list[Job]:
    document = tomllib.loads(text)
    raw_jobs = document.get('jobs')
    if not isinstance(raw_jobs, list):
        raise TypeError('top-level jobs must be an array of tables')

    jobs: list[Job] = []
    failures: list[Exception] = []

    for index, raw in enumerate(raw_jobs):
        try:
            if not isinstance(raw, dict):
                raise TypeError('job must be a table')

            missing = {'id', 'operation', 'left'} - raw.keys()
            if missing:
                raise KeyError(f'missing required keys: {sorted(missing)}')

            job_id = raw['id']
            if not isinstance(job_id, str) or not job_id.strip():
                raise TypeError('id must be a non-empty string')

            try:
                operation = Operation(str(raw['operation']).lower())
            except ValueError as exc:
                exc.add_note('allowed_operations=' + ','.join(op.value for op in Operation))
                raise

            left = raw['left']
            if type(left) not in {int, float}:
                raise TypeError('left must be numeric')

            right = raw.get('right')
            if right is not None and type(right) not in {int, float}:
                raise TypeError('right must be numeric when present')

            delay_ms = raw.get('delay_ms', 0)
            if type(delay_ms) is not int or delay_ms < 0:
                raise ValueError('delay_ms must be a non-negative integer')

            if operation in {Operation.ADD, Operation.DIVIDE} and right is None:
                raise ValueError(f'{operation.value} requires right')
            if operation is Operation.SQRT and right is not None:
                raise ValueError('sqrt does not accept right')

            jobs.append(
                Job(
                    id=job_id,
                    operation=operation,
                    left=float(left),
                    right=None if right is None else float(right),
                    delay_ms=delay_ms,
                )
            )
        except (KeyError, TypeError, ValueError) as exc:
            exc.add_note(f'job_index={index}')
            if isinstance(raw, dict):
                exc.add_note(f"job_id={raw.get('id', '<missing>')!r}")
            failures.append(exc)

    if failures:
        raise ExceptionGroup('job configuration is invalid', failures)

    return jobs

In [43]:
async def execute_job(job: Job) -> JobResult:
    try:
        await asyncio.sleep(job.delay_ms / 1000)

        match job.operation:
            case Operation.ADD:
                assert job.right is not None
                value = job.left + job.right
            case Operation.DIVIDE:
                assert job.right is not None
                value = job.left / job.right
            case Operation.SQRT:
                value = math.sqrt(job.left)
            case _ as unreachable:
                assert_never(unreachable)

        return JobResult(job.id, value)
    except (ArithmeticError, ValueError) as exc:
        exc.add_note(f'job_id={job.id!r}')
        exc.add_note(f'operation={job.operation.value!r}')
        raise


async def run_jobs(jobs: list[Job], *, timeout_seconds: float) -> list[JobResult]:
    results: list[JobResult | None] = [None] * len(jobs)

    async def run_one(index: int, job: Job) -> None:
        results[index] = await execute_job(job)

    async with asyncio.timeout(timeout_seconds):
        async with asyncio.TaskGroup() as group:
            for index, job in enumerate(jobs):
                group.create_task(run_one(index, job), name=f'job:{job.id}')

    return [result for result in results if result is not None]

In [44]:
CAPSTONE_TOML = r"""
[[jobs]]
id = "sum"
operation = "add"
left = 10.5
right = -10.504

[[jobs]]
id = "ratio"
operation = "divide"
left = 9
right = 3

[[jobs]]
id = "root"
operation = "sqrt"
left = 81
"""

jobs = parse_jobs(CAPSTONE_TOML)
results = await run_jobs(jobs, timeout_seconds=1.0)

for result in results:
    print(result.id, result.formatted())

assert [result.id for result in results] == ['sum', 'ratio', 'root']
assert results[0].formatted() == '-0.004'
assert results[1].formatted() == '3.000'
assert results[2].formatted() == '9.000'

sum -0.004
ratio 3.000
root 9.000


In [45]:
FAILING_CAPSTONE_TOML = r"""
[[jobs]]
id = "divide-by-zero"
operation = "divide"
left = 1
right = 0

[[jobs]]
id = "negative-root"
operation = "sqrt"
left = -1
"""

failing_jobs = parse_jobs(FAILING_CAPSTONE_TOML)
execution_messages: list[str] = []

try:
    await run_jobs(failing_jobs, timeout_seconds=1.0)
except* ZeroDivisionError as group:
    for leaf in iter_leaf_exceptions(group):
        execution_messages.append(f'ZERO DIVISION: {leaf}; notes={leaf.__notes__}')
except* ValueError as group:
    for leaf in iter_leaf_exceptions(group):
        execution_messages.append(f'VALUE ERROR: {leaf}; notes={leaf.__notes__}')

print('\n'.join(execution_messages))
assert len(execution_messages) == 2

ZERO DIVISION: float division by zero; notes=["job_id='divide-by-zero'", "operation='divide'"]
VALUE ERROR: math domain error; notes=["job_id='negative-root'", "operation='sqrt'"]


In [46]:
INVALID_CAPSTONE_TOML = r"""
[[jobs]]
id = "missing-right"
operation = "add"
left = 10

[[jobs]]
id = "bad-op"
operation = "multiply"
left = 2
right = 3

[[jobs]]
id = "bad-delay"
operation = "sqrt"
left = 4
delay_ms = -10
"""

configuration_messages: list[str] = []
try:
    parse_jobs(INVALID_CAPSTONE_TOML)
except* (KeyError, TypeError, ValueError) as group:
    for leaf in iter_leaf_exceptions(group):
        configuration_messages.append(
            f'{type(leaf).__name__}: {leaf}; notes={getattr(leaf, "__notes__", [])}'
        )

print('\n'.join(configuration_messages))
assert len(configuration_messages) == 3

ValueError: add requires right; notes=['job_index=0', "job_id='missing-right'"]
ValueError: 'multiply' is not a valid Operation; notes=['allowed_operations=add,divide,sqrt', 'job_index=1', "job_id='bad-op'"]
ValueError: delay_ms must be a non-negative integer; notes=['job_index=2', "job_id='bad-delay'"]


# 12. Review checklist

Use this checklist when adopting Python 3.11 features in production:

- **Tracebacks:** retain source files and line information in deployed artifacts.
- **Exception notes:** add context, but do not place secrets in exceptions or logs.
- **Exception groups:** introduce them deliberately and document them as part of the API.
- **`except*`:** handle only the leaf types you can actually recover from or report.
- **`TaskGroup`:** treat sibling cancellation as part of the structured-concurrency contract.
- **Timeouts:** decide whether the timeout applies to each operation or the whole group.
- **TOML:** validate parsed values; parsing alone does not enforce your application schema.
- **Typing:** run a static type checker in CI; annotations alone do not provide runtime validation.
- **`StrEnum`:** be aware of strict `type(x) is str` consumers.
- **Regex atomicity:** confirm semantics before optimizing backtracking.
- **Huge integers:** enforce an application-level limit even when the interpreter also has one.
- **Performance:** benchmark your actual workload and Python patch release.

# 13. Further practice problems

These are intentionally left without solutions in the main flow so the notebook can also be used as an assessment.

1. Extend the capstone with a `power` operation and maintain exhaustive matching.
2. Add per-job timeouts while retaining the overall timeout.
3. Create an exception-group summarizer that counts leaves by concrete exception type.
4. Validate unknown TOML keys and report all of them together.
5. Add a retry layer that records each attempt with `add_note()` and raises all final failures together.
6. Build a `StrEnum`-based HTTP method router.
7. Write a regex benchmark comparing a naive nested quantifier, an atomic group, and a possessive quantifier.
8. Add a static-checker-friendly SQL builder using `LiteralString` and parameter placeholders.
9. Create a variadic generic `Record` whose `map()` operation preserves tuple arity.
10. Inspect specialization before and after calling the same function with different operand types.

## Compact answer key for further practice

Suggested implementation directions:

1. Add `POWER = auto()`, require `right`, and add one `match` arm.
2. Wrap each `execute_job(job)` call in its own `asyncio.timeout(job_timeout)`.
3. Recursively traverse `BaseExceptionGroup.exceptions` and accumulate a `Counter`.
4. Compare each table's keys with an allowed-key set and collect `ValueError`s.
5. Catch the operation error per attempt, add an attempt note, and store it; raise an `ExceptionGroup` after retries are exhausted.
6. Normalize user input, construct the enum, and dispatch through exhaustive matching.
7. Use bounded input sizes and `timeit`; never run an unbounded catastrophic-backtracking example.
8. Keep SQL structure literal and pass user values separately as parameters.
9. Parameterize the record with `TypeVarTuple`; preserve `tuple[Unpack[Ts]]` in its public methods.
10. Warm up each operand pattern separately and use `dis.dis(..., adaptive=True, show_caches=True)`.